<a href="https://colab.research.google.com/github/k246151droid/lyrank-ml-internship/blob/main/digital_twin_on_renewable_energy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
#         backend
###            ----------------matlab code and simullink code----------
''' %% ========================================================================
%  PROJECT: Edge-AI Digital Twin Solar Inverter Simulation
%  STEP 1: Automated Simulink Model Builder and Sensor Data Generator
% ========================================================================
clear; clc; close all;

modelName = 'Solar_Inverter_DigitalTwin';

% Check if model is already open, close without saving
if bdIsLoaded(modelName)
    close_system(modelName, 0);
end

% Create a new Simulink System
new_system(modelName);
open_system(modelName);

fprintf('--> Building Simulink Model: %s...\n', modelName);

%% 1. ADD SYSTEM BLOCKS PROGRAMMATICALLY
% Solar PV Array & DC Link
add_block('simulink/Sources/Constant', [modelName '/Solar_Irradiance'], 'Value', '1000', 'Position', [50, 50, 110, 80]);
add_block('simulink/Sources/Sine Wave', [modelName '/AC_Grid_Ref'], 'Amplitude', '311', 'Frequency', '2*pi*50', 'Position', [50, 130, 110, 160]);

% Fault Injection Switches (Dynamic System Faults)
add_block('simulink/Sources/Step', [modelName '/Capacitor_Degradation_Fault'], 'Time', '2.5', 'Before', '1.5', 'After', '12.5', 'Position', [50, 210, 110, 240]);
add_block('simulink/Sources/Step', [modelName '/Thermal_Overheat_Fault'], 'Time', '4.0', 'Before', '45.0', 'After', '85.0', 'Position', [50, 290, 110, 320]);

% Physical System Dynamics Subsystems (Transfer Functions)
add_block('simulink/Continuous/Transfer Fcn', [modelName '/DC_Link_Capacitor'], 'Numerator', '[1]', 'Denominator', '[0.001 1]', 'Position', [200, 50, 280, 90]);
add_block('simulink/Continuous/Transfer Fcn', [modelName '/IGBT_Inverter_Bridge'], 'Numerator', '[10]', 'Denominator', '[0.0001 1]', 'Position', [200, 130, 280, 170]);

% Mathematical Sensor Measurements Aggregator
add_block('simulink/Math Operations/Gain', [modelName '/V_dc_Sensor'], 'Gain', '600', 'Position', [340, 50, 380, 80]);
add_block('simulink/Math Operations/Gain', [modelName '/I_alpha_Sensor'], 'Gain', '50', 'Position', [340, 130, 380, 160]);

% Bus Creator for Multi-Channel Telemetry (6 Features)
add_block('simulink/Signal Routing/Bus Creator', [modelName '/Sensor_Bus'], 'Inputs', '6', 'Position', [450, 40, 455, 340]);

% To Workspace Block for Data Export
add_block('simulink/Sinks/To Workspace', [modelName '/Sensor_Data_Export'], 'VariableName', 'sensor_telemetry', 'SaveFormat', 'Timeseries', 'Position', [520, 175, 590, 205]);

%% 2. CONNECT BLOCKS WITH SIGNAL LINES
add_line(modelName, 'Solar_Irradiance/1', 'DC_Link_Capacitor/1');
add_line(modelName, 'AC_Grid_Ref/1', 'IGBT_Inverter_Bridge/1');

add_line(modelName, 'DC_Link_Capacitor/1', 'V_dc_Sensor/1');
add_line(modelName, 'IGBT_Inverter_Bridge/1', 'I_alpha_Sensor/1');

% Connect Sensors to Bus Creator
add_line(modelName, 'V_dc_Sensor/1', 'Sensor_Bus/1');
add_line(modelName, 'Capacitor_Degradation_Fault/1', 'Sensor_Bus/2'); % V_ripple
add_line(modelName, 'I_alpha_Sensor/1', 'Sensor_Bus/3');
add_line(modelName, 'AC_Grid_Ref/1', 'Sensor_Bus/4'); % I_beta
add_line(modelName, 'Thermal_Overheat_Fault/1', 'Sensor_Bus/5'); % Temp_j
add_line(modelName, 'DC_Link_Capacitor/1', 'Sensor_Bus/6'); % P_ac Proxy

% Connect Bus Creator to Workspace Export
add_line(modelName, 'Sensor_Bus/1', 'Sensor_Data_Export/1');

% Format Layout Automatically
Simulink.BlockDiagram.arrangeSystem(modelName);
save_system(modelName);
fprintf('✅ Simulink Model Successfully Built & Saved as %s.slx!\n', modelName);

%% 3. RUN AUTOMATED SIMULATION & EXPORT CSV DATASET
fprintf('--> Running Simulation for 6.0 seconds (Injecting Faults at t=2.5s and t=4.0s)...\n');

set_param(modelName, 'StopTime', '6.0');
simResults = sim(modelName);

% Extract Data Arrays
time_vec = simResults.sensor_telemetry.Time;
data_matrix = simResults.sensor_telemetry.Data;

% Create Table matching Python Feature Schema: [V_dc, V_ripple, I_alpha, I_beta, Temp_j, P_ac]
featureTable = array2table(data_matrix, ...
    'VariableNames', {'V_dc', 'V_ripple', 'I_alpha', 'I_beta', 'Temp_j', 'P_ac'});

% Add Timestamp
featureTable.Time = time_vec;

% Save to CSV File for Python TinyML
csvFileName = 'inverter_sensor_dataset.csv';
writetable(featureTable, csvFileName);

fprintf('✅ Simulation Finished! Sensor Dataset exported to %s (%d rows).\n', csvFileName, height(featureTable));

%% 4. PLOT TELEMETRY & FAULT INJECTION GRAPH
figure('Name', 'Inverter Telemetry with Injected Faults', 'Color', 'w');
subplot(3,1,1);
plot(time_vec, data_matrix(:,1), 'b', 'LineWidth', 1.5);
ylabel('V_{dc} (Volts)'); title('DC-Link Voltage'); grid on;

subplot(3,1,2);
plot(time_vec, data_matrix(:,2), 'r', 'LineWidth', 1.5);
xline(2.5, '--k', 'Fault: ESR Ripple Surge');
ylabel('V_{ripple} (Volts)'); title('Capacitor Ripple (Degradation Fault at t=2.5s)'); grid on;

subplot(3,1,3);
plot(time_vec, data_matrix(:,5), 'm', 'LineWidth', 1.5);
xline(4.0, '--k', 'Fault: IGBT Overheat');
ylabel('Temp_j (\circC)'); xlabel('Time (seconds)'); title('Junction Temperature (Overheat Fault at t=4.0s)'); grid on;'''
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

print("🚀 Step 1: Training TinyML Autoencoder Model...")

# 1. Dataset Generator (Normal / Healthy Telemetry)
def generate_inverter_data(num_samples=5000):
    np.random.seed(42)
    V_dc = np.random.normal(loc=600.0, scale=2.0, size=num_samples)
    V_ripple = np.random.normal(loc=1.5, scale=0.2, size=num_samples)
    I_alpha = np.random.normal(loc=50.0, scale=1.0, size=num_samples)
    I_beta = np.random.normal(loc=50.0, scale=1.0, size=num_samples)
    Temp_j = np.random.normal(loc=45.0, scale=1.5, size=num_samples)
    P_ac = np.random.normal(loc=30000.0, scale=100.0, size=num_samples)

    healthy_data = np.column_stack((V_dc, V_ripple, I_alpha, I_beta, Temp_j, P_ac))
    mean = np.mean(healthy_data, axis=0)
    std = np.std(healthy_data, axis=0)
    return (healthy_data - mean) / std, mean, std

X_train, mean, std = generate_inverter_data(num_samples=5000)

# 2. Autoencoder Architecture
input_layer = layers.Input(shape=(X_train.shape[1],))
encoded = layers.Dense(16, activation='relu')(input_layer)
bottleneck = layers.Dense(8, activation='relu')(encoded)
decoded = layers.Dense(16, activation='relu')(bottleneck)
output_layer = layers.Dense(X_train.shape[1], activation='linear')(decoded)

autoencoder = models.Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.fit(X_train, X_train, epochs=15, batch_size=32, verbose=0)

# 3. Full INT8 Quantization (TinyML Optimization)
def representative_dataset_gen():
    for i in range(100):
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(autoencoder)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model_quant = converter.convert()

# Save Model File
with open('inverter_tinyml_model.tflite', 'wb') as f:
    f.write(tflite_model_quant)

print(f"✅ Backend Done: TinyML INT8 Model Saved Successfully! (Size: {len(tflite_model_quant) / 1024:.2f} KB)")

🚀 Step 1: Training TinyML Autoencoder Model...
Saved artifact at '/tmp/tmpafr6mb_d'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 6), dtype=tf.float32, name='keras_tensor_10')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  139827192075024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139827192079440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139827192082896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139827192073296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139827192077520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139827192076368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139827192074064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139827192075984: TensorSpec(shape=(), dtype=tf.resource, name=None)
✅ Backend Done: TinyML INT8 Model Saved Successfully! (Size: 4.27 KB)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [10]:
# dsh board
%%writefile dashboard.py
import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.subplots as sp
import streamlit as st

st.set_page_config(page_title="Edge-AI Digital Twin", page_icon="⚡", layout="wide")
st.title("⚡ Edge-AI Digital Twin: Solar Inverter Control Room")
st.markdown("**Fast-Path Latency:** `< 10ms` (Deterministic Protection) | **Slow-Path Interface:** Local WebSocket / MQTT")

# Sidebar Controls
st.sidebar.header("🕹️ Inverter Controls & Fault Injection")
operating_mode = st.sidebar.radio("Operating State", ["Normal (Healthy)", "Inject Capacitor Aging Fault", "Inject IGBT Thermal Fault"])
threshold = st.sidebar.slider("Anomaly Threshold (γ_th)", 0.1, 2.0, 0.5, 0.05)

# Session state initialization for real-time history
if "history" not in st.session_state:
    st.session_state.history = {"Time": [], "V_dc": [], "V_ripple": [], "Temp_j": [], "Loss": []}

def generate_telemetry(mode):
    V_dc = 600.0 + np.random.normal(0, 1.5)
    V_ripple = 1.5 + np.random.normal(0, 0.1)
    I_alpha = 50.0 + np.random.normal(0, 0.8)
    Temp_j = 45.0 + np.random.normal(0, 0.5)
    P_ac = 30000.0 + np.random.normal(0, 80.0)

    if mode == "Inject Capacitor Aging Fault":
        V_ripple += np.random.uniform(8.0, 14.0)
        V_dc -= np.random.uniform(10.0, 35.0)
    elif mode == "Inject IGBT Thermal Fault":
        Temp_j += np.random.uniform(35.0, 50.0)
        I_alpha += np.random.uniform(20.0, 40.0)

    baseline = np.array([600.0, 1.5, 50.0, 45.0, 30000.0])
    current = np.array([V_dc, V_ripple, I_alpha, Temp_j, P_ac])
    weights = np.array([0.05, 1.2, 0.3, 0.8, 0.01])
    rec_loss = float(np.mean(np.square((current - baseline) / baseline) * weights * 100))
    return V_dc, V_ripple, Temp_j, P_ac, rec_loss

# Get telemetry data step
V_dc, V_ripple, Temp_j, P_ac, rec_loss = generate_telemetry(operating_mode)
curr_time = time.strftime("%H:%M:%S")

st.session_state.history["Time"].append(curr_time)
st.session_state.history["V_dc"].append(V_dc)
st.session_state.history["V_ripple"].append(V_ripple)
st.session_state.history["Temp_j"].append(Temp_j)
st.session_state.history["Loss"].append(rec_loss)

# Retain last 30 data points
for k in st.session_state.history:
    st.session_state.history[k] = st.session_state.history[k][-30:]

is_anomaly = rec_loss > threshold

# Dynamic Metrics Display
col1, col2, col3, col4 = st.columns(4)
col1.metric("DC Bus Voltage", f"{V_dc:.1f} V")
col2.metric("DC Ripple", f"{V_ripple:.2f} V")
col3.metric("Junction Temp", f"{Temp_j:.1f} °C")
with col4:
    if is_anomaly: st.error("🚨 TRIP TRIGGERED")
    else: st.success("🟢 NORMAL STATE")

# Real-time Multi-plot Grid
fig = sp.make_subplots(rows=2, cols=2, subplot_titles=("TinyML Anomaly Loss", "Voltage Ripple", "Junction Temperature", "DC-Bus Voltage"))
hist = st.session_state.history

fig.add_trace(go.Scatter(x=hist["Time"], y=hist["Loss"], line=dict(color="red" if is_anomaly else "green")), row=1, col=1)
fig.add_hline(y=threshold, line_dash="dash", line_color="orange", row=1, col=1)
fig.add_trace(go.Scatter(x=hist["Time"], y=hist["V_ripple"], line=dict(color="orange")), row=1, col=2)
fig.add_trace(go.Scatter(x=hist["Time"], y=hist["Temp_j"], line=dict(color="purple")), row=2, col=1)
fig.add_trace(go.Scatter(x=hist["Time"], y=hist["V_dc"], line=dict(color="blue")), row=2, col=2)
fig.update_layout(height=480, showlegend=False)

st.plotly_chart(fig, use_container_width=True)

time.sleep(1)
st.rerun()

Overwriting dashboard.py


In [ ]:
#cloutfare
# 1. Necessary Python Packages
!pip install -q streamlit plotly pandas numpy

# 2. Install Cloudflare Official Binary
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 3. Kill existing process
!pkill -f streamlit

# 4. Launch Dashboard & Generate Tunnel Link
import os, time
os.system("streamlit run dashboard.py --server.port 8501 > /dev/null 2>&1 &")
time.sleep(3)

print("🚀 Dashboard live ho raha hai! Neechay walay Cloudflare Link par click karein:\n")
!cloudflared tunnel --url http://localhost:8501

🚀 Dashboard live ho raha hai! Neechay walay Cloudflare Link par click karein:

2026-07-24T07:54:23Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-07-24T07:54:23Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-07-24T07:54:27Z INF +--------------------------------------------------------------------------------------------+
2026-07-24T07:54:27Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
20